# Can we predict failure within 90 days of an inspection?
Target: `failed_90d`. Approach: check data quality, look for leakage, split **by time** (the model would be used on future inspections), and compare simple and boosted models against a base-rate baseline.

In [ ]:
import warnings, os
os.environ['LOKY_MAX_CPU_COUNT'] = '4'
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit

df = pd.read_csv('data.csv', parse_dates=['inspected_at']).sort_values('inspected_at').reset_index(drop=True)
print(df.shape, 'missing:', int(df.isna().sum().sum()), 'dup rows:', int(df.duplicated().sum()), 'unique assets:', df.asset_tag.nunique())
print('failure rate:', df.failed_90d.mean().round(4), '| span:', df.inspected_at.min(), '->', df.inspected_at.max())
df.describe().T

## 1. Leakage check: `parts_replaced`
The dictionary says this is the parts count on the work order *that closed this inspection record*. That work order is completed after the inspection, so it can reflect a failure/repair that happened in the 90-day window. Let's see.

In [ ]:
df.groupby('parts_replaced').failed_90d.agg(['mean', 'size'])

Every record with 2+ parts replaced failed, and none with 0 did: it is essentially a restatement of the label, not something known at inspection time. **It is excluded from the features.** Section 6 shows how much it would (misleadingly) inflate the score.

## 2. Failure rate over time and by site

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
q = df.groupby(df.inspected_at.dt.to_period('Q')).failed_90d.mean()
q.index = q.index.astype(str)
q.plot.bar(ax=ax[0], color='#4C78A8'); ax[0].set_title('Failure rate by inspection quarter'); ax[0].set_ylabel('rate')
df.groupby('site').failed_90d.mean().plot.bar(ax=ax[1], color='#4C78A8'); ax[1].set_title('Failure rate by site')
plt.tight_layout(); plt.show()

The rate swings a lot by quarter (about 3% to 27%), while sites look alike (~12%). Consequences: a time-based split is the honest evaluation, and the test period has a lower base rate than training, so absolute precision will look low. The latest quarters may also be partly **right-censored** (failures after the data extract aren't observed yet), which would deflate recent rates; worth confirming with the data owner.

## 3. Signal in the inspection measurements

In [ ]:
num = ['vibration_rms', 'bearing_temp_c', 'ambient_temp_c', 'meter_hours', 'load_pct']
fig, axs = plt.subplots(1, 5, figsize=(15, 3))
for a, col in zip(axs, num):
    b = pd.qcut(df[col], 8, duplicates='drop')
    df.groupby(b, observed=True).failed_90d.mean().reset_index(drop=True).plot(ax=a, marker='o', color='#4C78A8')
    a.set_title(col, fontsize=9); a.set_xlabel('octile'); a.axhline(df.failed_90d.mean(), ls=':', c='grey')
axs[0].set_ylabel('failure rate'); plt.tight_layout(); plt.show()

## 4. Time-based split (first 70% of inspections for training, latest 30% for testing)

In [ ]:
feats = num + ['site']
X = pd.get_dummies(df[feats], columns=['site'], dtype=float)
y = df.failed_90d
cut = int(len(df) * .7)
Xtr, Xte, ytr, yte = X.iloc[:cut], X.iloc[cut:], y.iloc[:cut], y.iloc[cut:]
print('train', df.inspected_at.iloc[0].date(), '->', df.inspected_at.iloc[cut-1].date(), 'rate', ytr.mean().round(3))
print('test ', df.inspected_at.iloc[cut].date(), '->', df.inspected_at.iloc[-1].date(), 'rate', yte.mean().round(3))
models = {
    'logistic': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    'grad. boosting': HistGradientBoostingClassifier(max_iter=150, learning_rate=.05, max_depth=3, random_state=0),
}
rows, probs = [], {}
for n, m in models.items():
    m.fit(Xtr, ytr); p = m.predict_proba(Xte)[:, 1]; probs[n] = p
    rows.append((n, roc_auc_score(yte, p), average_precision_score(yte, p)))
rows.append(('base rate (no skill)', 0.5, yte.mean()))
pd.DataFrame(rows, columns=['model', 'ROC AUC', 'PR AUC']).set_index('model').round(3)

PR AUC for a no-skill model equals the test base rate, so that is the number to beat.

In [ ]:
r = pd.DataFrame({'y': yte.values, 'p': probs['grad. boosting']})
r['decile'] = pd.qcut(r.p.rank(method='first'), 10, labels=False) + 1
lift = r.groupby('decile').y.agg(['mean', 'sum']).rename(columns={'mean': 'failure_rate', 'sum': 'failures'})
print('Top-decile lift vs base rate: %.1fx' % (lift.failure_rate.iloc[-1] / yte.mean()))
top20 = r.sort_values('p', ascending=False).head(int(len(r) * .2)).y.sum() / r.y.sum()
print('Share of test failures caught in the top 20%% of risk scores: %.0f%%' % (100 * top20))
lift.round(3)

In [ ]:
pi = permutation_importance(models['grad. boosting'], Xte, yte, scoring='roc_auc', n_repeats=10, random_state=0)
pd.Series(pi.importances_mean, X.columns).sort_values().plot.barh(color='#4C78A8', figsize=(6, 3.5), title='Permutation importance (drop in test ROC AUC)')
plt.tight_layout(); plt.show()

## 5. Stability across forward-in-time folds

In [ ]:
res = []
for i, (a, b) in enumerate(TimeSeriesSplit(5).split(X)):
    m = HistGradientBoostingClassifier(max_iter=150, learning_rate=.05, max_depth=3, random_state=0).fit(X.iloc[a], y.iloc[a])
    p = m.predict_proba(X.iloc[b])[:, 1]
    res.append((i, df.inspected_at.iloc[b[0]].date(), y.iloc[b].mean(), roc_auc_score(y.iloc[b], p), average_precision_score(y.iloc[b], p)))
pd.DataFrame(res, columns=['fold', 'test_start', 'base_rate', 'ROC AUC', 'PR AUC']).round(3)

## 6. What the leaky feature would have done

In [ ]:
Xl = X.assign(parts_replaced=df.parts_replaced)
m = HistGradientBoostingClassifier(max_iter=150, learning_rate=.05, max_depth=3, random_state=0).fit(Xl.iloc[:cut], ytr)
p = m.predict_proba(Xl.iloc[cut:])[:, 1]
print('WITH parts_replaced (leaky): ROC AUC %.3f, PR AUC %.3f' % (roc_auc_score(yte, p), average_precision_score(yte, p)))

## Conclusions (from a run of the cells above)
- **Yes, modestly.** On the latest 30% of inspections, gradient boosting reaches ROC AUC ~0.76 and PR AUC ~0.16 against a 0.056 base rate (logistic: 0.68 / 0.10). The top 20% of risk scores contain ~55% of failures; the top decile fails ~2.8x the base rate.
- Held up across 5 forward-in-time folds (AUC 0.66-0.85, weakest on the smallest early training set), but precision is low: this ranks risk, it doesn't pinpoint failures.
- `parts_replaced` is leakage (AUC 0.998 with it) and must not be used.
- Open questions: the large quarter-to-quarter swing in failure rate and possible right-censoring of recent inspections; each asset appears once, so asset history features aren't available.
- Outputs were not saved in the file; re-run all cells to populate them.